# VRP SOFR EOD Update v1 — Step 2: Validated production write

The check-only test passed:

- 6 new observations
- 0 revisions
- earliest changed date: `2026-07-02`
- refreshed maximum date: `2026-07-10`
- all hard checks passed

This version keeps the same checks and adds a **controlled production write**.

The notebook will:

1. download and validate FRED SOFR;
2. compare refreshed data with the canonical cache;
3. create timestamped audit files;
4. create a timestamped backup of the existing cache;
5. write the refreshed data to a temporary file;
6. validate the temporary file;
7. atomically replace the canonical cache;
8. re-read and validate the final canonical file.

The production-write switch is deliberately set to `False` initially. Run the check cells first. Then change it to `True` and run only the final write cell.


In [1]:
from __future__ import annotations

import io
import json
import os
import shutil
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from IPython.display import display


FRED_SERIES_ID = "SOFR"
FRED_CSV_URL = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={FRED_SERIES_ID}"


def resolve_project_root() -> Path:
    """Locate the VRP project without relying solely on the current directory."""
    candidates: list[Path] = []

    env_root = os.getenv("VRP_PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend(
        [
            Path.home() / "vrp_project",
            Path(r"C:\Users\patri\vrp_project"),
        ]
    )

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.resolve()
        checked.append(str(candidate))
        cache = candidate / "data" / "external" / "fred_sofr_history.csv"
        if cache.exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the VRP project root. Checked:\n- "
        + "\n- ".join(dict.fromkeys(checked))
        + "\n\nSet VRP_PROJECT_ROOT or run from inside "
          "C:\\Users\\patri\\vrp_project."
    )


PROJECT_ROOT = resolve_project_root()
CACHE_PATH = PROJECT_ROOT / "data" / "external" / "fred_sofr_history.csv"
AUDIT_ROOT = PROJECT_ROOT / "data" / "audit" / "sofr_eod_update_v1"

print("=" * 100)
print("VRP SOFR EOD update v1 — Step 2")
print("=" * 100)
print(f"Project root: {PROJECT_ROOT}")
print(f"Cache path:   {CACHE_PATH}")
print(f"Audit root:   {AUDIT_ROOT}")
print(f"FRED source:  {FRED_CSV_URL}")


VRP SOFR EOD update v1 — Step 2
Project root: C:\Users\patri\vrp_project
Cache path:   C:\Users\patri\vrp_project\data\external\fred_sofr_history.csv
Audit root:   C:\Users\patri\vrp_project\data\audit\sofr_eod_update_v1
FRED source:  https://fred.stlouisfed.org/graph/fredgraph.csv?id=SOFR


In [2]:
def normalize_sofr_frame(raw: pd.DataFrame, source_name: str) -> pd.DataFrame:
    """Normalize a cache or FRED download to the canonical two-column schema."""
    if raw.empty:
        raise ValueError(f"{source_name} returned an empty table.")

    columns_by_lower = {str(col).strip().lower(): col for col in raw.columns}

    date_col = next(
        (
            columns_by_lower[name]
            for name in ("observation_date", "date")
            if name in columns_by_lower
        ),
        None,
    )
    value_col = next(
        (
            columns_by_lower[name]
            for name in ("sofr", "value")
            if name in columns_by_lower
        ),
        None,
    )

    if date_col is None or value_col is None:
        raise ValueError(
            f"{source_name} does not contain recognizable SOFR columns. "
            f"Columns received: {list(raw.columns)}"
        )

    out = raw[[date_col, value_col]].copy()
    out.columns = ["observation_date", "SOFR"]
    out["observation_date"] = pd.to_datetime(
        out["observation_date"], errors="coerce"
    ).dt.normalize()
    out["SOFR"] = pd.to_numeric(out["SOFR"], errors="coerce")

    out = (
        out.dropna(subset=["observation_date", "SOFR"])
        .sort_values("observation_date")
        .drop_duplicates(subset=["observation_date"], keep="last")
        .reset_index(drop=True)
    )
    return out


def download_fred_sofr() -> pd.DataFrame:
    """Download the official FRED SOFR CSV."""
    response = requests.get(
        FRED_CSV_URL,
        headers={
            "User-Agent": "vrp-sofr-eod-update-v1/1.0",
            "Accept": "text/csv,application/csv,*/*",
        },
        timeout=30,
    )
    response.raise_for_status()

    if not response.content:
        raise RuntimeError("FRED returned an empty response body.")

    return normalize_sofr_frame(
        pd.read_csv(io.StringIO(response.text)),
        "FRED download",
    )


def build_comparison(
    existing: pd.DataFrame,
    refreshed: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.Timestamp]:
    comparison = existing.merge(
        refreshed,
        on="observation_date",
        how="outer",
        suffixes=("_old", "_fresh"),
        indicator=True,
    )

    new_rows = (
        comparison.loc[
            comparison["_merge"].eq("right_only"),
            ["observation_date", "SOFR_fresh"],
        ]
        .rename(columns={"SOFR_fresh": "SOFR"})
        .sort_values("observation_date")
        .reset_index(drop=True)
    )

    missing_from_fred = (
        comparison.loc[
            comparison["_merge"].eq("left_only"),
            ["observation_date", "SOFR_old"],
        ]
        .rename(columns={"SOFR_old": "SOFR"})
        .sort_values("observation_date")
        .reset_index(drop=True)
    )

    both_mask = comparison["_merge"].eq("both")
    revised_mask = both_mask & ~np.isclose(
        comparison["SOFR_old"],
        comparison["SOFR_fresh"],
        rtol=0.0,
        atol=1e-12,
        equal_nan=True,
    )

    revised_rows = (
        comparison.loc[
            revised_mask,
            ["observation_date", "SOFR_old", "SOFR_fresh"],
        ]
        .sort_values("observation_date")
        .reset_index(drop=True)
    )

    changed_dates = pd.concat(
        [
            new_rows[["observation_date"]],
            revised_rows[["observation_date"]],
        ],
        ignore_index=True,
    ).drop_duplicates()

    first_changed = (
        changed_dates["observation_date"].min()
        if not changed_dates.empty
        else pd.NaT
    )

    return comparison, new_rows, revised_rows, missing_from_fred, first_changed


def validate_sofr(
    existing: pd.DataFrame,
    refreshed: pd.DataFrame,
    missing_from_fred: pd.DataFrame,
) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "check": "canonical_columns_exact",
                "severity": "hard",
                "passed": list(refreshed.columns)
                == ["observation_date", "SOFR"],
                "detail": str(list(refreshed.columns)),
            },
            {
                "check": "download_not_empty",
                "severity": "hard",
                "passed": len(refreshed) > 0,
                "detail": f"rows={len(refreshed):,}",
            },
            {
                "check": "trade_dates_unique",
                "severity": "hard",
                "passed": refreshed["observation_date"].is_unique,
                "detail": (
                    f"duplicate_rows="
                    f"{int(refreshed['observation_date'].duplicated().sum()):,}"
                ),
            },
            {
                "check": "dates_monotonic",
                "severity": "hard",
                "passed": refreshed["observation_date"].is_monotonic_increasing,
                "detail": "sorted ascending",
            },
            {
                "check": "rates_finite",
                "severity": "hard",
                "passed": bool(np.isfinite(refreshed["SOFR"]).all()),
                "detail": (
                    f"nonfinite_rows="
                    f"{int((~np.isfinite(refreshed['SOFR'])).sum()):,}"
                ),
            },
            {
                "check": "rates_within_sanity_range",
                "severity": "hard",
                "passed": bool(refreshed["SOFR"].between(-5.0, 25.0).all()),
                "detail": (
                    f"min={refreshed['SOFR'].min():.6f}; "
                    f"max={refreshed['SOFR'].max():.6f}; units=percent"
                ),
            },
            {
                "check": "fresh_max_not_older_than_cache",
                "severity": "hard",
                "passed": (
                    refreshed["observation_date"].max()
                    >= existing["observation_date"].max()
                ),
                "detail": (
                    f"old_max={existing['observation_date'].max().date()}; "
                    f"fresh_max={refreshed['observation_date'].max().date()}"
                ),
            },
            {
                "check": "fresh_history_not_truncated",
                "severity": "hard",
                "passed": len(refreshed) >= len(existing),
                "detail": (
                    f"old_rows={len(existing):,}; "
                    f"fresh_rows={len(refreshed):,}"
                ),
            },
            {
                "check": "no_existing_dates_missing_from_fred",
                "severity": "hard",
                "passed": missing_from_fred.empty,
                "detail": f"missing_dates={len(missing_from_fred):,}",
            },
        ]
    )


existing = normalize_sofr_frame(pd.read_csv(CACHE_PATH), "Existing cache")
refreshed = download_fred_sofr()

(
    comparison,
    new_rows,
    revised_rows,
    missing_from_fred,
    first_changed_sofr_date,
) = build_comparison(existing, refreshed)

validations = validate_sofr(existing, refreshed, missing_from_fred)
hard_checks_passed = bool(
    validations.loc[validations["severity"].eq("hard"), "passed"].all()
)

print("=" * 100)
print("Comparison summary")
print("=" * 100)
print(f"Existing rows:          {len(existing):,}")
print(f"Refreshed rows:         {len(refreshed):,}")
print(f"New observations:       {len(new_rows):,}")
print(f"Revised observations:   {len(revised_rows):,}")
print(f"Old dates absent FRED:  {len(missing_from_fred):,}")
print(
    "Earliest changed date: "
    + (
        str(first_changed_sofr_date.date())
        if pd.notna(first_changed_sofr_date)
        else "None"
    )
)
print(
    f"Old max / fresh max:    "
    f"{existing['observation_date'].max().date()} / "
    f"{refreshed['observation_date'].max().date()}"
)

print("\nValidation")
display(validations)
print(f"\nHARD CHECKS PASSED: {hard_checks_passed}")

print("\nLatest refreshed rows")
display(refreshed.tail(10))

print("\nNew observations")
display(new_rows)

print("\nRevised observations")
display(revised_rows)

if not missing_from_fred.empty:
    print("\nExisting observations absent from refreshed FRED history")
    display(missing_from_fred.head(20))


Comparison summary
Existing rows:          2,059
Refreshed rows:         2,065
New observations:       6
Revised observations:   0
Old dates absent FRED:  0
Earliest changed date: 2026-07-02
Old max / fresh max:    2026-07-01 / 2026-07-10

Validation


,check,severity,passed,detail
0,canonical_columns_exact,hard,True,"['observation_date', 'SOFR']"
1,download_not_empty,hard,True,"rows=2,065"
2,trade_dates_unique,hard,True,duplicate_rows=0
3,dates_monotonic,hard,True,sorted ascending
4,rates_finite,hard,True,nonfinite_rows=0
5,rates_within_sanity_range,hard,True,min=0.010000; max=5.400000; units=percent
6,fresh_max_not_older_than_cache,hard,True,old_max=2026-07-01; fresh_max=2026-07-10
7,fresh_history_not_truncated,hard,True,"old_rows=2,059; fresh_rows=2,065"
8,no_existing_dates_missing_from_fred,hard,True,missing_dates=0



HARD CHECKS PASSED: True

Latest refreshed rows


,observation_date,SOFR
2055,2026-06-26,3.62
2056,2026-06-29,3.62
2057,2026-06-30,3.68
2058,2026-07-01,3.66
2059,2026-07-02,3.64
2060,2026-07-06,3.63
2061,2026-07-07,3.62
2062,2026-07-08,3.58
2063,2026-07-09,3.53
2064,2026-07-10,3.55



New observations


,observation_date,SOFR
0,2026-07-02,3.64
1,2026-07-06,3.63
2,2026-07-07,3.62
3,2026-07-08,3.58
4,2026-07-09,3.53
5,2026-07-10,3.55



Revised observations


,observation_date,SOFR_old,SOFR_fresh


## Production-write control

Run the previous cells first.

Only after confirming `HARD CHECKS PASSED: True`, change:

```python
WRITE_PRODUCTION = False
```

to:

```python
WRITE_PRODUCTION = True
```

Then run the final cell **once**.


In [3]:
WRITE_PRODUCTION = True

if not WRITE_PRODUCTION:
    print(
        "WRITE_PRODUCTION is False. No files were changed.\n"
        "After reviewing the comparison and validation above, set it to True "
        "and rerun this cell."
    )
else:
    if not hard_checks_passed:
        failed = validations.loc[~validations["passed"]]
        raise RuntimeError(
            "SOFR production write blocked because hard validation failed:\n"
            + failed.to_string(index=False)
        )

    run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")
    run_dir = AUDIT_ROOT / run_timestamp
    run_dir.mkdir(parents=True, exist_ok=False)

    backup_path = CACHE_PATH.with_name(
        f"{CACHE_PATH.stem}.backup_{run_timestamp}{CACHE_PATH.suffix}"
    )
    temp_path = CACHE_PATH.with_name(
        f"{CACHE_PATH.stem}.tmp_{run_timestamp}{CACHE_PATH.suffix}"
    )

    validation_path = run_dir / "sofr_validation.csv"
    new_rows_path = run_dir / "sofr_new_observations.csv"
    revised_rows_path = run_dir / "sofr_revised_observations.csv"
    missing_path = run_dir / "sofr_existing_dates_missing_from_fred.csv"
    refreshed_snapshot_path = run_dir / "fred_sofr_history_refreshed_snapshot.csv"
    manifest_path = run_dir / "sofr_update_manifest.json"

    # Persist audit evidence before touching the canonical file.
    validations.to_csv(validation_path, index=False)
    new_rows.to_csv(new_rows_path, index=False)
    revised_rows.to_csv(revised_rows_path, index=False)
    missing_from_fred.to_csv(missing_path, index=False)
    refreshed.to_csv(refreshed_snapshot_path, index=False)

    # Backup current canonical cache.
    shutil.copy2(CACHE_PATH, backup_path)

    # Write the candidate cache to a temporary file in the same directory.
    refreshed.to_csv(
        temp_path,
        index=False,
        date_format="%Y-%m-%d",
        lineterminator="\n",
    )

    # Re-read and validate the temporary file before atomic replacement.
    temp_reloaded = normalize_sofr_frame(
        pd.read_csv(temp_path),
        "Temporary candidate",
    )
    temp_validations = validate_sofr(existing, temp_reloaded, missing_from_fred)
    temp_hard_passed = bool(
        temp_validations.loc[
            temp_validations["severity"].eq("hard"), "passed"
        ].all()
    )

    exact_candidate_match = bool(
        temp_reloaded.equals(
            refreshed.reset_index(drop=True)
        )
    )

    if not temp_hard_passed or not exact_candidate_match:
        temp_path.unlink(missing_ok=True)
        raise RuntimeError(
            "Temporary SOFR candidate failed pre-publication validation. "
            f"temp_hard_passed={temp_hard_passed}; "
            f"exact_candidate_match={exact_candidate_match}"
        )

    # Atomic replacement: canonical is changed only after all checks above pass.
    os.replace(temp_path, CACHE_PATH)

    # Re-read the final canonical file and verify exact equality.
    canonical_after = normalize_sofr_frame(
        pd.read_csv(CACHE_PATH),
        "Canonical cache after publication",
    )
    final_exact_match = bool(
        canonical_after.equals(
            refreshed.reset_index(drop=True)
        )
    )

    if not final_exact_match:
        # Restore the backup if the post-publication readback is not exact.
        shutil.copy2(backup_path, CACHE_PATH)
        raise RuntimeError(
            "Post-publication SOFR readback did not match the validated "
            "candidate. The prior canonical cache was restored."
        )

    manifest = {
        "status": "PUBLISHED",
        "source": "FRED_SOFR",
        "source_url": FRED_CSV_URL,
        "run_timestamp_utc": run_timestamp,
        "project_root": str(PROJECT_ROOT),
        "canonical_path": str(CACHE_PATH),
        "backup_path": str(backup_path),
        "audit_directory": str(run_dir),
        "old_rows": int(len(existing)),
        "new_rows_total": int(len(refreshed)),
        "rows_added": int(len(new_rows)),
        "rows_revised": int(len(revised_rows)),
        "old_dates_absent_from_fred": int(len(missing_from_fred)),
        "old_min_date": str(existing["observation_date"].min().date()),
        "old_max_date": str(existing["observation_date"].max().date()),
        "new_min_date": str(refreshed["observation_date"].min().date()),
        "new_max_date": str(refreshed["observation_date"].max().date()),
        "first_changed_sofr_date": (
            str(first_changed_sofr_date.date())
            if pd.notna(first_changed_sofr_date)
            else None
        ),
        "hard_checks_passed": hard_checks_passed,
        "post_publish_exact_match": final_exact_match,
        "audit_files": {
            "validation": str(validation_path),
            "new_observations": str(new_rows_path),
            "revised_observations": str(revised_rows_path),
            "missing_from_fred": str(missing_path),
            "refreshed_snapshot": str(refreshed_snapshot_path),
        },
    }

    manifest_path.write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )

    print("=" * 100)
    print("SOFR PRODUCTION UPDATE PUBLISHED")
    print("=" * 100)
    print(f"Canonical cache:          {CACHE_PATH}")
    print(f"Backup:                   {backup_path}")
    print(f"Audit directory:          {run_dir}")
    print(f"Rows before / after:      {len(existing):,} / {len(refreshed):,}")
    print(f"Rows added:               {len(new_rows):,}")
    print(f"Rows revised:             {len(revised_rows):,}")
    print(
        "First changed SOFR date: "
        + (
            str(first_changed_sofr_date.date())
            if pd.notna(first_changed_sofr_date)
            else "None"
        )
    )
    print(f"Latest SOFR date:         {canonical_after['observation_date'].max().date()}")
    print(f"Latest SOFR rate:         {canonical_after.iloc[-1]['SOFR']:.4f}%")
    print(f"Post-publish exact match: {final_exact_match}")


SOFR PRODUCTION UPDATE PUBLISHED
Canonical cache:          C:\Users\patri\vrp_project\data\external\fred_sofr_history.csv
Backup:                   C:\Users\patri\vrp_project\data\external\fred_sofr_history.backup_20260713_231215_UTC.csv
Audit directory:          C:\Users\patri\vrp_project\data\audit\sofr_eod_update_v1\20260713_231215_UTC
Rows before / after:      2,059 / 2,065
Rows added:               6
Rows revised:             0
First changed SOFR date: 2026-07-02
Latest SOFR date:         2026-07-10
Latest SOFR rate:         3.5500%
Post-publish exact match: True


## Expected successful result

The final output should show:

- `SOFR PRODUCTION UPDATE PUBLISHED`
- rows before / after: `2,059 / 2,065`
- rows added: `6`
- rows revised: `0`
- first changed SOFR date: `2026-07-02`
- latest SOFR date: `2026-07-10`
- latest SOFR rate: `3.5500%`
- post-publish exact match: `True`

After that succeeds, the next task is to modify the EOD pipeline so that:

1. this SOFR updater runs before implied variance;
2. `2026-07-02` becomes the upstream recalculation start;
3. implied variance and downstream signals are rebuilt from that date;
4. the health check compares SOFR with the expected published observation date rather than the target trade date.
